# verify: Interactive Visual Explorer

> ASR asks "what did they say?" Speaker recognition asks "who said it?" The math looks the same — embeddings plus cosine — but every production decision hinges on a single EER number.

Welcome to the interactive companion notebook for **verify**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Speaker verification: toy MFCC-stat embeddings, cosine scoring, EER.

Synthetic "speakers" are sinusoid mixtures with different harmonic profiles.
We enroll each speaker, build same/diff trial pairs, compute the EER.

Run: python3 code/main.py
"""

import math
import random
from collections import defaultdict

def tone_mix(freqs, sr, seconds, amp=0.4, noise=0.02):
    n = int(sr * seconds)
    out = []
    for i in range(n):
        val = 0.0
        t = i / sr
        for f in freqs:
            val += math.sin(2.0 * math.pi * f * t)
        val = amp * val / max(1, len(freqs))
        val += random.gauss(0, noise)
        out.append(val)
    return out


In [ ]:
def hann(N):
    return [0.5 * (1.0 - math.cos(2.0 * math.pi * n / (N - 1))) for n in range(N)]

def dft_mag(x):
    n = len(x)
    half = n // 2 + 1
    out = []
    for k in range(half):
        re = 0.0
        im = 0.0
        for j in range(n):
            angle = -2.0 * math.pi * k * j / n
            re += x[j] * math.cos(angle)
            im += x[j] * math.sin(angle)
        out.append(math.sqrt(re * re + im * im))
    return out


In [ ]:
def frame_signal(sig, frame_len, hop):
    n = 1 + max(0, (len(sig) - frame_len) // hop)
    return [sig[i * hop : i * hop + frame_len] for i in range(n)]

def stft_mag(sig, frame_len, hop):
    w = hann(frame_len)
    frames = frame_signal(sig, frame_len, hop)
    return [dft_mag([w[j] * f[j] for j in range(frame_len)]) for f in frames]

def hz_to_mel(f):
    return 2595.0 * math.log10(1.0 + f / 700.0)


In [ ]:
def mel_to_hz(m):
    return 700.0 * (10 ** (m / 2595.0) - 1.0)

def mel_filterbank(n_mels, n_fft, sr):
    fmin, fmax = 0.0, sr / 2
    mels = [hz_to_mel(fmin) + (hz_to_mel(fmax) - hz_to_mel(fmin)) * i / (n_mels + 1) for i in range(n_mels + 2)]
    hzs = [mel_to_hz(m) for m in mels]
    half = n_fft // 2 + 1
    bins = [min(half - 1, int(round(h * n_fft / sr))) for h in hzs]
    fb = [[0.0] * half for _ in range(n_mels)]
    for m in range(n_mels):
        left, center, right = bins[m], bins[m + 1], bins[m + 2]
        for k in range(left, center):
            fb[m][k] = (k - left) / max(1, center - left)
        for k in range(center, right):
            fb[m][k] = (right - k) / max(1, right - center)
    return fb


In [ ]:
def apply_filterbank(spec, fb):
    return [[sum(w * frame[k] for k, w in enumerate(f) if w) for f in fb] for frame in spec]

def log_transform(x, eps=1e-10):
    return [[math.log(max(v, eps)) for v in row] for row in x]

def dct_ii(x, n_coeffs):
    N = len(x)
    return [sum(x[n] * math.cos(math.pi * k * (2 * n + 1) / (2 * N)) for n in range(N)) for k in range(n_coeffs)]

def featurize(signal, sr, n_mfcc=13, n_mels=40, frame_len=256, hop=128):
    mag = stft_mag(signal, frame_len, hop)
    fb = mel_filterbank(n_mels, frame_len, sr)
    mels = apply_filterbank(mag, fb)
    lm = log_transform(mels)
    return [dct_ii(f, n_mfcc) for f in lm]


In [ ]:
def l2_normalize(v):
    norm = math.sqrt(sum(x * x for x in v)) or 1e-12
    return [x / norm for x in v]

def embed_mfcc_stats(signal, sr):
    frames = featurize(signal, sr)
    n = len(frames[0])
    mean = [sum(f[i] for f in frames) / len(frames) for i in range(n)]
    var = [sum((f[i] - mean[i]) ** 2 for f in frames) / len(frames) for i in range(n)]
    std = [math.sqrt(v) for v in var]
    return l2_normalize(mean + std)


In [ ]:
def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

def eer(same_scores, diff_scores):
    thresholds = sorted(set(same_scores + diff_scores))
    best_gap = float("inf")
    best_fa, best_fr, best_t = 1.0, 0.0, thresholds[0] if thresholds else 0.0
    for t in thresholds:
        fr = sum(1 for s in same_scores if s < t) / len(same_scores)
        fa = sum(1 for s in diff_scores if s >= t) / len(diff_scores)
        gap = abs(fa - fr)
        if gap < best_gap:
            best_gap = gap
            best_fa, best_fr, best_t = fa, fr, t
    return (best_fa + best_fr) / 2, best_t


In [ ]:
def main():
    random.seed(123)
    sr = 8000
    duration = 0.4

speakers = {
        "alice": [200, 400, 600],
        "bob":   [220, 330, 880],
        "carol": [300, 600, 1200],
        "dave":  [180, 540, 1080],
        "eve":   [260, 520, 780],
    }

n_per = 5
    print("=== Enroll 5 synthetic speakers, 5 utterances each ===")
    enroll = defaultdict(list)
    for spk, freqs in speakers.items():
        for _ in range(n_per):
            sig = tone_mix(freqs, sr, duration, noise=0.04)
            enroll[spk].append(embed_mfcc_stats(sig, sr))
        print(f"  {spk}: {len(enroll[spk])} embeddings, dim={len(enroll[spk][0])}")


In [ ]:
print()
    print("=== Build trial pairs (same vs different speaker) ===")
    same_scores = []
    diff_scores = []
    spk_list = list(speakers.keys())
    for spk in spk_list:
        embs = enroll[spk]
        for i in range(len(embs)):
            for j in range(i + 1, len(embs)):
                same_scores.append(cosine(embs[i], embs[j]))
    for i, s1 in enumerate(spk_list):
        for s2 in spk_list[i + 1:]:
            for e1 in enroll[s1]:
                for e2 in enroll[s2]:
                    diff_scores.append(cosine(e1, e2))
    print(f"  same-speaker pairs: {len(same_scores)}  mean cosine: {sum(same_scores)/len(same_scores):.3f}")
    print(f"  diff-speaker pairs: {len(diff_scores)}  mean cosine: {sum(diff_scores)/len(diff_scores):.3f}")


In [ ]:
print()
    print("=== Equal Error Rate ===")
    e, t = eer(same_scores, diff_scores)
    print(f"  EER: {e * 100:.2f}%   at threshold: {t:.3f}")
    print(f"  synthetic speakers are near-orthogonal, so this toy hits 0% EER.")
    print(f"  real ECAPA-TDNN on VoxCeleb1-O lands at 0.87% after training on 2700 speakers.")

print()
    print("=== 2026 speaker-verification leaderboard ===")
    table = [
        ("ReDimNet (2024)",       0.39, "24M"),
        ("WavLM-SV large",        0.42, "316M"),
        ("Pyannote 3.1",          0.65, "6M"),
        ("ECAPA-TDNN",            0.87, "15M"),
        ("x-vector (classic)",    3.10, "5M"),
    ]
    print("  | Model              | EER  | Params |")
    for name, e, p in table:
        print(f"  | {name:<18} | {e:.2f} | {p:<6} |")


In [ ]:
if __name__ == "__main__":
    main()
